# Building an Image Classifier with PyTorch

## Using TorchVision to Load the Dataset

Zoe Buck

I used Claude to create all the code in this notebook via separate prompts. I combined all of the resulting scripts in this notebook, and included each prompt before each section of code. 

Note: I just use Claude and not Claude Code, since Claude Code is not included in the free version. 

**Prompt:** "Using torchvision provide me a python scrip to load the FashionMNIST dataset and splkit it into training (55,000) and validation (5,000) sets."

In [4]:
"""
Load FashionMNIST with torchvision and split into training (55,000)
and validation (5,000) sets.
"""
 
import torch
from torch.utils.data import random_split, DataLoader
import torchvision
import torchvision.transforms.v2 as transforms
import torchmetrics

 
# ---- Transform: convert PIL images to float32 tensors and normalize ----
transform = transforms.Compose([
    transforms.ToImage(),                              # PIL -> tv_tensor Image (uint8)
    transforms.ToDtype(torch.float32, scale=True),      # uint8 -> float32 in [0, 1]
    transforms.Normalize((0.2860,), (0.3530,))          # FashionMNIST mean/std
])
 
# ---- Download / load the full training set (60,000 images) ----
full_train_dataset = torchvision.datasets.FashionMNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
 
# ---- Load the official test set (10,000 images) ----
test_dataset = torchvision.datasets.FashionMNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)
 
# ---- Split the 60,000 training images into 55,000 train / 5,000 val ----
train_size = 55_000
val_size = 5_000
assert train_size + val_size == len(full_train_dataset)
 
generator = torch.Generator().manual_seed(42)  # for reproducibility
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=generator
)
 
# ---- Wrap in DataLoaders ----
torch.manual_seed(42)
batch_size = 64
 
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


**Prompt:** "Show the shape and data type of the x sample from the training data. Also the text label attached to the y sample indexes in the full dataset."

In [ ]:
X_sample, y_sample = train_dataset[0]

In [6]:
X_sample.shape

torch.Size([1, 28, 28])

In [7]:
X_sample.dtype

torch.float32

In [8]:
full_train_dataset.classes[y_sample]

'Ankle boot'

## Building the Classifier

**Prompt:** "Also help me build an image classifier class. Create a model, and xentropy variable."

In [9]:
# ---- Model definition ----
class ImageClassifier(torch.nn.Module):
    """Simple CNN for classifying 28x28 grayscale FashionMNIST images into 10 classes."""
 
    def __init__(self, num_classes: int = 10):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 32, kernel_size=3, padding=1),  # 28x28 -> 28x28
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),                              # -> 14x14
            torch.nn.Conv2d(32, 64, kernel_size=3, padding=1),  # -> 14x14
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),                              # -> 7x7
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(64 * 7 * 7, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, num_classes),
        )
 
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
 
 
# ---- Instantiate model, move to device, define loss ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ImageClassifier(num_classes=10).to(device)
xentropy = torch.nn.CrossEntropyLoss()


**Prompt:** "Now create an optimizer and compute the accuracy from the train2 function (which I have attached for context). Next, switch to evaluation mode and extract the evaluation data. Get a predicted y value (index of the largest logit). Print out the values at the predicted indexes in full_train_dataset. Check if it made the correct predictions."

In [10]:
# ---- Optimizer ----
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
 
# ---- Metric ----
metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
 
 
def evaluate_tm(model, loader, metric):
    """Run inference over a full loader and return the computed metric."""
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()
 
 
def train2(model, optimizer, criterion, metric, train_loader, valid_loader,
           n_epochs):
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.
        metric.reset()
        for X_batch, y_batch in train_loader:
            model.train()
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
        mean_loss = total_loss / len(train_loader)
        history["train_losses"].append(mean_loss)
        history["train_metrics"].append(metric.compute().item())
        history["valid_metrics"].append(
            evaluate_tm(model, valid_loader, metric).item())
        print(f"Epoch {epoch + 1}/{n_epochs}, "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.4f}, "
              f"valid metric: {history['valid_metrics'][-1]:.4f}")
    return history

print(f"Full training set size: {len(full_train_dataset)}")
print(f"Train subset size:      {len(train_dataset)}")
print(f"Validation subset size: {len(val_dataset)}")
print(f"Test set size:          {len(test_dataset)}")

images, labels = next(iter(train_loader))
print(f"\nSample batch shape: {images.shape}")
print(f"Sample batch dtype:  {images.dtype}")
print(f"Sample labels shape: {labels.shape}")

print(f"\nDevice: {device}")
print(model)

# Quick sanity check: forward pass + loss computation
images, labels = images.to(device), labels.to(device)
outputs = model(images)
loss = xentropy(outputs, labels)
print(f"\nOutput shape: {outputs.shape}")
print(f"Initial loss (untrained): {loss.item():.4f}")

# ---- Run training over multiple epochs ----
n_epochs = 5
history = train2(model, optimizer, xentropy, metric,
                    train_loader, val_loader, n_epochs)

# ---- Switch to evaluation mode ----
model.eval()

# ---- Extract a batch of evaluation (validation) data ----
eval_images, eval_labels = next(iter(val_loader))
eval_images, eval_labels = eval_images.to(device), eval_labels.to(device)

# ---- Get predicted y value: index of the largest logit ----
with torch.no_grad():
    eval_outputs = model(eval_images)
    predicted = eval_outputs.argmax(dim=1)

# ---- Print the class names at the predicted indexes ----
class_names = full_train_dataset.classes
predicted_labels = [class_names[idx] for idx in predicted.tolist()]
actual_labels = [class_names[idx] for idx in eval_labels.tolist()]

print("\nPredicted classes:", predicted_labels)
print("Actual classes:   ", actual_labels)

# ---- Check if predictions were correct ----
correct_mask = predicted == eval_labels
num_correct = correct_mask.sum().item()
print(f"\nCorrect predictions: {num_correct}/{len(eval_labels)}")
print("Correct? ", correct_mask.tolist())


Full training set size: 60000
Train subset size:      55000
Validation subset size: 5000
Test set size:          10000

Sample batch shape: torch.Size([64, 1, 28, 28])
Sample batch dtype:  torch.float32
Sample labels shape: torch.Size([64])

Device: cuda
ImageClassifier(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


MIOpen(HIP): Warning [OpenRuntimeLibraryForDevice] CK grouped conv library not found for device gfx1030: libMIOpenCKGroupedConv_gfx1030.so: cannot open shared object file: No such file or directory



Output shape: torch.Size([64, 10])
Initial loss (untrained): 2.2891
Epoch 1/5, train loss: 0.4755, train metric: 0.8281, valid metric: 0.8782
Epoch 2/5, train loss: 0.3108, train metric: 0.8879, valid metric: 0.8986
Epoch 3/5, train loss: 0.2613, train metric: 0.9049, valid metric: 0.9032
Epoch 4/5, train loss: 0.2315, train metric: 0.9150, valid metric: 0.9128
Epoch 5/5, train loss: 0.2074, train metric: 0.9240, valid metric: 0.9196

Predicted classes: ['Sneaker', 'Coat', 'Pullover', 'Sandal', 'Ankle boot', 'Bag', 'Sneaker', 'Sneaker', 'Sneaker', 'Coat', 'Coat', 'Coat', 'Sneaker', 'Sneaker', 'Shirt', 'T-shirt/top', 'Trouser', 'Dress', 'Sneaker', 'Trouser', 'Pullover', 'Coat', 'Coat', 'T-shirt/top', 'Coat', 'Pullover', 'Coat', 'Bag', 'Sandal', 'Sneaker', 'Shirt', 'Ankle boot', 'Coat', 'T-shirt/top', 'Sandal', 'Coat', 'Sandal', 'Ankle boot', 'Bag', 'Bag', 'Coat', 'Coat', 'T-shirt/top', 'Sneaker', 'Dress', 'Pullover', 'Pullover', 'Sandal', 'Ankle boot', 'T-shirt/top', 'Pullover', 'Ankle

**Prompt:** "Use torch.nn.functional to apply a softmax function to the predicted y values (rounded to 3 decimal points)."

In [ ]:
import torch.nn.functional as F

# Convert logits to class probabilities
probabilities = F.softmax(eval_outputs, dim=1)
probabilities.round(decimals=3)

**Prompt:** "Apply a softmax function to the top 4 y values and indexes round the probabilities to 3 decimal points. Finally, calculate the total number of parameters."

In [ ]:
# Top-4 predicted classes and their softmax probabilities
top4_probs, top4_indexes = torch.topk(probabilities, k=4, dim=1)
top4_probs = torch.round(top4_probs * 1000) / 1000

print("Top-4 probabilities:\n", top4_probs)
print("Top-4 indexes:\n", top4_indexes)

# Total number of parameters in the model
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

tensor([[0.9590, 0.0400, 0.0000, 0.0000],
        [0.9930, 0.0070, 0.0000, 0.0000],
        [0.5820, 0.2310, 0.1840, 0.0030]], device='cuda:0')

**When I asked Claude for a summary of the conversation, the output was:**

"Summary of what we built:

Data loading — loaded FashionMNIST via torchvision.datasets.FashionMNIST and split the 60,000-image training set into 55,000 train / 5,000 validation samples with random_split (seeded for reproducibility), plus wrapped everything in DataLoaders.

Preprocessing — switched to torchvision.transforms.v2, using ToImage() + ToDtype(torch.float32, scale=True) to guarantee float32 tensors, followed by normalization.

Model — defined ImageClassifier, a small CNN (2 conv+pool blocks → FC head) for the 10 FashionMNIST classes.

Training setup — added an Adam optimizer, CrossEntropyLoss (xentropy), and a torchmetrics.Accuracy metric.

Training loop — implemented train2 to match a pattern you shared: it owns the epoch loop, logs a history dict (train_losses, train_metrics, valid_metrics), and validates each epoch via a companion evaluate_tm function.

Inference & evaluation — switched to model.eval(), pulled a validation batch, computed predictions via argmax, mapped predicted/actual indexes to class names, and checked correctness.

Softmax & top-k — applied torch.nn.functional.softmax to get class probabilities, then used torch.topk to get the top-4 predicted classes/probabilities, rounded to 3 decimals.

Parameter count — summed p.numel() over model.parameters() for the total parameter count."